# 07n — Trackintel end-to-end pipeline comparator

This stage intentionally changes the upstream implementation.

```text
raw GeoLife
   ↓
Trackintel read_geolife
   ↓
Trackintel sliding staypoints
   ↓
Trackintel DBSCAN locations
   ↓
per-stay IANA local wall clock
   ↓
Trackintel OSNA
   ↓
spatial / temporal comparison
   ↓
our frozen CP1 + CP2-v2
```

Primary staypoint thresholds are aligned to frozen CP1 so the first comparison asks whether a different implementation of the same broad stay definition produces similar events:

- distance: 200 m;
- dwell: 20 min;
- gap: 5 min.

Location clustering is deliberately Trackintel DBSCAN, evaluated at both `epsilon=100m` (library default) and `epsilon=200m` (radius sensitivity). `num_samples=1` is kept because Stage 07n is a pipeline comparator, not a DBSCAN MinPts tuning stage.

OSNA runs with `pre_filter=False` so semantic selection can be observed separately from extra Trackintel eligibility gates.

This notebook measures decomposition, not accuracy, and does not retune production.

In [ ]:
from pathlib import Path
import gc
import importlib.metadata
import importlib.util
import os
import shutil
import subprocess
import sys

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH=os.environ.get('GEOLIFE_REPO_BRANCH','main')
REPO_DIR=Path(os.environ.get('GEOLIFE_REPO_DIR','/root/geolife'))
V=Path('/mnt/geolife-data')
TRACKINTEL_VERSION='1.4.2'

def run(cmd,cwd=None):
    subprocess.run(cmd,cwd=cwd,check=True)

if not (REPO_DIR/'.git').exists():
    run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)])
else:
    run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH])
    run(['git','-C',str(REPO_DIR),'checkout',BRANCH])
    run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'])

run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR)

try:
    installed_ti=importlib.metadata.version('trackintel')
except importlib.metadata.PackageNotFoundError:
    installed_ti=None
geopandas_missing=importlib.util.find_spec('geopandas') is None
if installed_ti != TRACKINTEL_VERSION or geopandas_missing:
    print(
        f'Installing trackintel=={TRACKINTEL_VERSION} '
        f'(current={installed_ti}, geopandas_missing={geopandas_missing}) ...'
    )
    run([sys.executable,'-m','pip','install','-q','--upgrade',f'trackintel=={TRACKINTEL_VERSION}'])

for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path:
        sys.path.insert(0,p)

import geopandas as gpd
import numpy as np
import pandas as pd
import trackintel as ti
from timezonefinder import TimezoneFinder
from geolife.model import HomeOfficeConfig, build_semantic_locations, infer_home_office

spec=importlib.util.spec_from_file_location(
    's07n',
    REPO_DIR/'analysis'/'07n_trackintel_end_to_end.py',
)
assert spec is not None and spec.loader is not None
s07n=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=s07n
spec.loader.exec_module(s07n)

print('sha:',subprocess.run(
    ['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],
    capture_output=True,text=True,check=True
).stdout.strip())
print('trackintel:',importlib.metadata.version('trackintel'))
print(s07n.synthetic_self_check())

## 1. Raw release + frozen production gates

Trackintel's GeoLife reader expects the directory that directly contains `000/ ... 181/`.

The mounted release has already been measured in this project as:

- 182 users;
- 18,670 trajectory files;
- 24,876,978 GPS points.

The raw point count is re-accumulated from Trackintel's own reader while the per-user cache is built.

In [ ]:
def find_raw_root():
    env=os.environ.get('GEOLIFE_RAW_ROOT')
    candidates=([Path(env)] if env else []) + [
        V/'extracted'/'Geolife Trajectories 1.3'/'Data',
        V/'Data',
        V/'Geolife Trajectories 1.3'/'Data',
    ]
    for p in candidates:
        if p.exists() and (p/'000'/'Trajectory').exists():
            return p
    hits=sorted(V.glob('**/Geolife Trajectories 1.3/Data'))
    for p in hits:
        if (p/'000'/'Trajectory').exists():
            return p
    raise FileNotFoundError('GeoLife raw Data/ root not found; set GEOLIFE_RAW_ROOT')

def find_cache(name):
    candidates=[
        V/'cache'/'03a_user_behavior_deep_dive'/name,
        V/'artifacts'/'03a'/name,
        REPO_DIR/'artifacts'/'03a'/name,
        V/'cache'/'cp2_home_office'/name,
    ]
    for p in candidates:
        if p.exists():
            return p
    hits=sorted(V.glob(f'**/{name}'))
    if hits:
        return hits[0]
    raise FileNotFoundError(name)

RAW_ROOT=find_raw_root()
user_ids=sorted(
    p.name for p in RAW_ROOT.iterdir()
    if p.is_dir() and p.name.isdigit()
)
trajectory_files=sum(
    1 for uid in user_ids
    for _ in (RAW_ROOT/uid/'Trajectory').glob('*.plt')
)
assert len(user_ids)==182
assert trajectory_files==18670

STAYS_PATH=find_cache('stays_baseline_v1.pkl')
stays=pd.read_pickle(STAYS_PATH)
cfg=HomeOfficeConfig()
semantic_stays,production_locations=build_semantic_locations(stays,config=cfg)
production=infer_home_office(stays,config=cfg)
label_counts=production['label'].value_counts().to_dict()

assert len(stays)==5821
assert stays['user_id'].astype(str).nunique()==136
assert len(semantic_stays)==5821
assert len(production_locations)==2015
assert int(production_locations['stay_count'].ge(2).sum())==716
assert int(production_locations.loc[production_locations['stay_count'].ge(2),'user_id'].nunique())==104
assert label_counts.get('HOME',0)==27
assert label_counts.get('OFFICE',0)==16

raw_gate=pd.DataFrame([{
    'raw_root':str(RAW_ROOT),
    'raw_users':len(user_ids),
    'trajectory_files':trajectory_files,
    'cp1_stays':len(stays),
    'cp1_stay_users':stays['user_id'].astype(str).nunique(),
    'production_locations':len(production_locations),
    'production_HOME':label_counts.get('HOME',0),
    'production_OFFICE':label_counts.get('OFFICE',0),
}])
print('raw release + production parity: PASS')
display(raw_gate)

## 2. Resumable Trackintel raw → staypoints → DBSCAN

Processing is deliberately per-user so the notebook never holds all 24.9M raw points in memory.

Each user is read with **Trackintel's own** `read_geolife`. A temporary directory contains a symlink to exactly one raw user folder, preserving Trackintel's expected GeoLife directory structure.

Per-user results are cached under:

```text
/mnt/geolife-data/cache/cp2_v2/07n_trackintel_end_to_end/users/
```

A rerun skips completed users.

In [ ]:
C=V/'cache'/'cp2_v2'/'07n_trackintel_end_to_end'
USER_CACHE=C/'users'
USER_CACHE.mkdir(parents=True,exist_ok=True)

PIPELINE_TAG='ti142_d200_t20_g5_db100_200'
TMP_ROOT=Path('/tmp/geolife_trackintel_single_user')

def as_staypoints(frame):
    gdf=gpd.GeoDataFrame(
        frame.copy(),
        geometry=frame.geometry.name,
        crs=frame.crs,
    )
    return ti.Staypoints(gdf)

def with_location_id_column(locs):
    out=gpd.GeoDataFrame(
        locs.copy(),
        geometry=locs.geometry.name,
        crs=locs.crs,
    )
    if 'location_id' not in out.columns:
        out['location_id']=pd.to_numeric(out.index,errors='raise').astype(int)
    return out

def process_one_user(uid):
    cache_path=USER_CACHE/f'{uid}_{PIPELINE_TAG}.pkl'
    if cache_path.exists():
        return pd.read_pickle(cache_path), True

    if TMP_ROOT.exists():
        shutil.rmtree(TMP_ROOT)
    TMP_ROOT.mkdir(parents=True)
    os.symlink(RAW_ROOT/uid,TMP_ROOT/uid,target_is_directory=True)

    pfs,_=ti.io.read_geolife(str(TMP_ROOT),print_progress=False)
    raw_points=int(len(pfs))

    _,sp=pfs.generate_staypoints(
        method='sliding',
        dist_threshold=200,
        time_threshold=20,
        gap_threshold=5,
        include_last=False,
        exclude_duplicate_pfs=True,
        print_progress=False,
        n_jobs=1,
    )

    payload={'user_id':uid,'raw_points':raw_points,'staypoints':int(len(sp))}
    if len(sp):
        sp100,loc100=as_staypoints(sp).generate_locations(
            method='dbscan',
            epsilon=100,
            num_samples=1,
            distance_metric='haversine',
            agg_level='user',
            print_progress=False,
            n_jobs=1,
        )
        sp200,loc200=as_staypoints(sp).generate_locations(
            method='dbscan',
            epsilon=200,
            num_samples=1,
            distance_metric='haversine',
            agg_level='user',
            print_progress=False,
            n_jobs=1,
        )
        payload.update({
            'sp100':gpd.GeoDataFrame(sp100.copy(),geometry=sp100.geometry.name,crs=sp100.crs),
            'loc100':with_location_id_column(loc100),
            'sp200':gpd.GeoDataFrame(sp200.copy(),geometry=sp200.geometry.name,crs=sp200.crs),
            'loc200':with_location_id_column(loc200),
        })
    else:
        payload.update({'sp100':None,'loc100':None,'sp200':None,'loc200':None})

    partial=cache_path.with_suffix('.partial.pkl')
    pd.to_pickle(payload,partial)
    os.replace(partial,cache_path)
    del pfs,sp
    gc.collect()
    return payload, False

payloads=[]
cache_hits=0
for idx,uid in enumerate(user_ids,1):
    payload,hit=process_one_user(uid)
    payloads.append(payload)
    cache_hits += int(hit)
    if idx % 10 == 0 or idx==len(user_ids):
        print(f'{idx:3d}/{len(user_ids)} users | cache hits={cache_hits}')

raw_points_total=sum(int(p['raw_points']) for p in payloads)
assert raw_points_total==24876978
print('Trackintel raw point reconciliation: PASS',raw_points_total)

In [ ]:
def concat_geo(parts,geometry_name):
    frames=[p for p in parts if p is not None and len(p)]
    if not frames:
        return gpd.GeoDataFrame()
    merged=pd.concat(frames,ignore_index=True)
    return gpd.GeoDataFrame(merged,geometry=geometry_name,crs='EPSG:4326')

sp100=concat_geo([p['sp100'] for p in payloads],'geom')
loc100=concat_geo([p['loc100'] for p in payloads],'center')
sp200=concat_geo([p['sp200'] for p in payloads],'geom')
loc200=concat_geo([p['loc200'] for p in payloads],'center')

assert len(sp100)==len(sp200)
assert sp100['user_id'].astype(str).nunique()==sp200['user_id'].astype(str).nunique()

loc_table100=s07n.trackintel_location_table(sp100,loc100,'dbscan100')
loc_table200=s07n.trackintel_location_table(sp200,loc200,'dbscan200')

pipeline_summary=pd.DataFrame([
    {
        'variant':'dbscan100',
        'raw_points':raw_points_total,
        'staypoints':len(sp100),
        'staypoint_users':sp100['user_id'].nunique(),
        'locations':len(loc_table100),
        'recurring_locations':int(loc_table100['stay_count'].ge(2).sum()),
        'recurring_location_users':int(loc_table100.loc[loc_table100['stay_count'].ge(2),'user_id'].nunique()),
    },
    {
        'variant':'dbscan200',
        'raw_points':raw_points_total,
        'staypoints':len(sp200),
        'staypoint_users':sp200['user_id'].nunique(),
        'locations':len(loc_table200),
        'recurring_locations':int(loc_table200['stay_count'].ge(2).sum()),
        'recurring_location_users':int(loc_table200.loc[loc_table200['stay_count'].ge(2),'user_id'].nunique()),
    },
])
print('TRACKINTEL PIPELINE SUMMARY')
display(pipeline_summary)

## 3. Decomposition A — staypoint extraction

DBSCAN does not affect the staypoint inventory, so `dbscan100` is used as the representative Trackintel stay table.

A strong match requires:

- positive temporal overlap;
- at least 50% overlap relative to the shorter stay;
- stay centers within 200 m.

The comparison is reciprocal: CP1 → Trackintel and Trackintel → CP1.

In [ ]:
stay_match_private,stay_match_summary=s07n.compare_stay_inventories(
    stays,
    sp100,
)
print('STAY EXTRACTION MATCH')
display(stay_match_summary)

## 4. Decomposition B — location geometry

For every frozen production location, find the nearest Trackintel DBSCAN location for the same user.

This is not a one-to-one cluster identity assertion. It measures whether the different clustering pipelines create a spatially corresponding anchor.

Both DBSCAN radii are reported.

In [ ]:
location_match_rows=[]
location_summary_rows=[]
for variant,table in [('dbscan100',loc_table100),('dbscan200',loc_table200)]:
    matches=s07n.nearest_location_matches(production_locations,table)
    matches['variant']=variant
    summary=s07n.summarize_location_geometry(matches)
    summary['variant']=variant
    location_match_rows.append(matches)
    location_summary_rows.append(summary)

location_matches_private=pd.concat(location_match_rows,ignore_index=True)
location_geometry_summary=pd.concat(location_summary_rows,ignore_index=True)

print('LOCATION GEOMETRY MATCH')
display(location_geometry_summary)

## 5. Local-time semantic adapter for the Trackintel pipeline

The raw Trackintel reader parses GeoLife timestamps as timezone-aware UTC. Before OSNA, each detected staypoint center is independently resolved to an IANA timezone, then its physical UTC timestamp is converted to local wall clock and re-encoded as dummy UTC.

This is the same semantic-clock strategy audited in 07m, but now applied to Trackintel's own staypoint inventory.

Unresolved timezone staypoints are reported and excluded only from OSNA; they remain present in the stay/location decomposition above.

In [ ]:
tf=TimezoneFinder(in_memory=True)

def resolve_tzids(sp):
    values=[
        tf.timezone_at(lng=float(point.x),lat=float(point.y))
        for point in sp.geometry
    ]
    return pd.Series(values,index=sp.index,dtype='object')

tzids=resolve_tzids(sp100)
resolved_mask=tzids.notna()
timezone_summary=pd.DataFrame([{
    'trackintel_staypoints':len(sp100),
    'timezone_resolved':int(resolved_mask.sum()),
    'timezone_unresolved':int((~resolved_mask).sum()),
    'resolved_users':int(sp100.loc[resolved_mask,'user_id'].nunique()),
    'distinct_timezones':int(tzids.loc[resolved_mask].nunique()),
}])
display(timezone_summary)

assert len(sp100)==len(sp200)
assert np.array_equal(
    sp100['started_at'].astype(str).to_numpy(),
    sp200['started_at'].astype(str).to_numpy(),
)
assert np.allclose(sp100.geometry.x.to_numpy(),sp200.geometry.x.to_numpy())
assert np.allclose(sp100.geometry.y.to_numpy(),sp200.geometry.y.to_numpy())

def local_trackintel_stays(sp):
    selected=sp.loc[resolved_mask].copy()
    selected_tz=tzids.loc[resolved_mask]
    adapted=s07n.adapt_trackintel_stays_to_local_wall(selected,selected_tz)
    return ti.Staypoints(gpd.GeoDataFrame(
        adapted,
        geometry=selected.geometry.name,
        crs=selected.crs,
    ))

local100=local_trackintel_stays(sp100)
local200=local_trackintel_stays(sp200)
print('Trackintel local-time semantic adapter: PASS')

## 6. Decomposition C — OSNA semantic candidates

OSNA now runs on Trackintel-generated staypoints and Trackintel-generated DBSCAN locations.

No Trackintel pre-filter is used:

```text
method='OSNA'
pre_filter=False
```

Because the location namespaces are different, semantic candidates are compared by center distance, not by integer `location_id`.

In [ ]:
candidate_frames=[]
semantic_comparisons=[]
for variant,local_sp,loc_table in [
    ('dbscan100',local100,loc_table100),
    ('dbscan200',local200,loc_table200),
]:
    labeled=ti.analysis.location_identifier(
        local_sp,
        method='OSNA',
        pre_filter=False,
    )
    candidates=s07n.extract_osna_candidates(labeled,loc_table,variant)
    candidate_frames.append(candidates)

trackintel_candidates=pd.concat(candidate_frames,ignore_index=True)

for variant in ['dbscan100','dbscan200']:
    comparison=s07n.compare_semantic_candidates(
        user_ids,
        production,
        trackintel_candidates,
        variant,
    )
    semantic_comparisons.append(comparison)

semantic_comparison_private=pd.concat(semantic_comparisons,ignore_index=True)
semantic_summary=s07n.summarize_semantic_candidates(semantic_comparison_private)
semantic_status=(
    semantic_comparison_private
    .groupby(['variant','label','status'],as_index=False)
    .size()
    .rename(columns={'size':'users'})
)

print('SEMANTIC CANDIDATE SPATIAL AGREEMENT')
display(semantic_summary)

print('SEMANTIC STATUS DISTRIBUTION')
display(semantic_status)

## 7. Decomposition snapshot

Read the three layers in order:

1. **Stay extraction** — do both pipelines detect temporally/spatially corresponding stays?
2. **Location geometry** — given their stay inventories, do their recurring anchors land near each other?
3. **Semantic selection** — given fully independent upstream pipelines, do HOME/WORK candidates land near production HOME/OFFICE?

A semantic disagreement cannot be attributed to OSNA until the upstream two layers are inspected.

In [ ]:
snapshot=pd.DataFrame([
    {
        'layer':'stay extraction',
        'metric':'CP1 strong match rate',
        'value':float(stay_match_summary.loc[stay_match_summary['source_inventory'].eq('CP1'),'strong_match_rate'].iloc[0]),
    },
    {
        'layer':'stay extraction',
        'metric':'Trackintel strong match rate',
        'value':float(stay_match_summary.loc[stay_match_summary['source_inventory'].eq('TRACKINTEL'),'strong_match_rate'].iloc[0]),
    },
    *[
        {
            'layer':'location geometry',
            'metric':f"{row.variant} recurring production locations within 200m",
            'value':(
                float(row.within_200m)/float(row.production_locations)
                if row.production_locations else np.nan
            ),
        }
        for row in location_geometry_summary.loc[
            location_geometry_summary['scope'].eq('recurring_production_locations')
        ].itertuples(index=False)
    ],
    *[
        {
            'layer':'semantic candidates',
            'metric':f"{row.variant} {row.label} within 200m / production emitted",
            'value':(
                float(row.within_200m)/float(row.production_emitted)
                if row.production_emitted else np.nan
            ),
        }
        for row in semantic_summary.itertuples(index=False)
    ],
])
display(snapshot)

## 8. Save private and aggregate outputs

Raw GPS points are never cached by this stage. Only per-user Trackintel-derived stay/location outputs and private comparison tables are stored on the Modal Volume.

In [ ]:
# Private comparison outputs
stay_match_private.to_pickle(C/'stay_inventory_matches_private.pkl')
location_matches_private.to_pickle(C/'location_geometry_matches_private.pkl')
trackintel_candidates.to_pickle(C/'trackintel_osna_candidates_private.pkl')
semantic_comparison_private.to_pickle(C/'semantic_candidate_comparison_private.pkl')

# Aggregate outputs
raw_gate.to_csv(C/'raw_and_production_gate.csv',index=False)
pipeline_summary.to_csv(C/'trackintel_pipeline_summary.csv',index=False)
stay_match_summary.to_csv(C/'stay_match_summary.csv',index=False)
location_geometry_summary.to_csv(C/'location_geometry_summary.csv',index=False)
timezone_summary.to_csv(C/'timezone_adapter_summary.csv',index=False)
semantic_summary.to_csv(C/'semantic_candidate_summary.csv',index=False)
semantic_status.to_csv(C/'semantic_status_distribution.csv',index=False)
snapshot.to_csv(C/'decomposition_snapshot.csv',index=False)

print('saved:',C)

## Interpretation boundary

- Stage 07n is an **end-to-end implementation comparator**, not semantic ground truth.
- Trackintel staypoint thresholds are aligned to CP1 at 200 m / 20 min / 5 min so stay-extraction differences primarily reflect implementation details rather than deliberately different thresholds.
- DBSCAN 100 m and 200 m are both reported because DBSCAN chaining and radius differ from production complete-link max-diameter 200 m.
- `num_samples=1` is not claimed optimal.
- OSNA receives per-stay local wall-clock time derived from Trackintel staypoint centers.
- Integer location ids are never compared across pipelines.
- Candidate distances of 50/100/200 m are descriptive spatial correspondence thresholds, not correctness thresholds.
- A difference in end-to-end HOME/OFFICE can arise from stay extraction, location clustering, semantic selection, or a combination.
- No Stage-07n output changes the frozen HOME 27 / OFFICE 16 policy.